In [ ]:
# run_raster_parallel.py
from __future__ import annotations

import os
import multiprocessing as mp
import concurrent.futures as cf
from pathlib import Path

from general_utils import find_ephys_sessions
from raster_worker import process_session, OUTDIR  # import the top-level worker

def main():
    OUTDIR.mkdir(parents=True, exist_ok=True)
    _, _, sessions = find_ephys_sessions()
    print("Sessions:", sessions)
    if not sessions:
        return

    # Use spawn to avoid HDF5/NWB fork-safety issues
    ctx = mp.get_context("spawn")
    max_workers = min(len(sessions), os.cpu_count() or 1)
    print(f"Running in parallel with {max_workers} workers (spawn)")

    with cf.ProcessPoolExecutor(max_workers=max_workers, mp_context=ctx) as ex:
        futures = {ex.submit(process_session, s): s for s in sessions}
        for fut in cf.as_completed(futures):
            print(fut.result())

if __name__ == "__main__":
    # If something else already set the start method, this is fine.
    try:
        mp.set_start_method("spawn", force=False)
    except RuntimeError:
        pass
    main()
